# qdrant 구축 및 접속 확인

In [2]:
from qdrant_client import QdrantClient

client = QdrantClient(host="localhost", port=6333)

print(client.get_collections())
print("접속 성공")

collections=[]
접속 성공


## 파일 경로 설정

In [5]:
from langchain_community.document_loaders import JSONLoader

# 진짜 청크 오면 여기만 바꾸기
CHUNK_PATH = "./data/sample_policy_chunks.jsonl"
EMB_PATH = "./data/policy_embeddings.jsonl"
COLLECTION_NAME = "policy_chunks_test"

## 청크 읽기
> 청크 파일에서 **정책 문장(page_content)**과 **꼬리표(metadata: 지역코드, 정책번호, 출처, 검수상태 등)**를 읽어 `documents`에 담는다.
>> → Qdrant에서 **검색 결과로 보여줄 내용**과 **필터 조건**으로 사용된다.

In [ ]:
def metadata_func(record: dict, metadata: dict) -> dict:
    for key, value in record.items():
        if key != "text":
            metadata[key] = value
    return metadata

loader = JSONLoader(
    file_path=CHUNK_PATH,
    jq_schema=".",
    content_key="text",
    json_lines=True,
    metadata_func=metadata_func,
)

documents = loader.load()
print(f"청크 수: {len(documents)}")
print(documents[0].page_content)
print(documents[0].metadata)

C:\Users\Playdata\AppData\Local\Temp\ipykernel_4204\1462714360.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import JSONLoader


청크 수: 6
[경기 수원시] 독거노인 생활지원 서비스. 대상: 수원시에 사는 65세 이상 혼자 사는 어르신. 내용: 생활관리사가 안부 확인과 생활 지원을 합니다. 신청: 주소지 행정복지센터.
{'source': 'C:\\dev\\SKN36-3rd-3Team\\notebooks\\qdrant\\data\\sample_policy_chunks.jsonl', 'seq_num': 1, 'chunk_id': 'POL-41110-0001-c0', 'policy_id': 'POL-41110-0001', 'document_id': None, 'section_id': None, 'region_code': '41110', 'region_name': '경기도 수원시', 'region_scope': 'sigungu', 'year': 2026, 'source_type': 'local_api', 'source_name': '지자체복지서비스 API', 'page': None, 'source_url': 'https://example.go.kr/policy/41110-0001', 'review_status': 'approved'}


## 벡터 파일 읽기
> 01에서 정책 문장을 임베딩해 저장한 **벡터(숫자 1024개)**를 읽어 `{chunk_id: 벡터}` 형태로 `vectors`에 담는다.
>> → Qdrant에서 **의미 검색에 쓰는 좌표**가 된다.
>>> 모델 이름과 벡터 길이(DIM)도 함께 읽어서, 질문 임베딩 모델과 컬렉션 크기를 맞춘다.

In [4]:
import json

vectors = {}
with open(EMB_PATH, encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        vectors[row["chunk_id"]] = row["vector"]

EMBED_MODEL = row["model"]
DIM = row["dim"]

print(f"벡터 수: {len(vectors)} / 모델: {EMBED_MODEL} / 길이: {DIM}")

# 청크는 있는데 벡터가 없는 게 있는지 검사
for doc in documents:
    if doc.metadata["chunk_id"] not in vectors:
        print("벡터 없음:", doc.metadata["chunk_id"])

벡터 수: 6 / 모델: qwen3-embedding:0.6b / 길이: 1024


## 컬렉션 만들기
> 데이터를 넣을 **빈 저장 공간(책장)**을 만든다.
>> 벡터 길이(size=1024)와 유사도 측정 방식(COSINE)을 정해야 벡터를 저장하고 비교할 수 있다.

In [6]:
from qdrant_client.models import VectorParams, Distance

# 연습용: 이미 있으면 지우고 새로 만들기
if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=DIM, distance=Distance.COSINE),
)

print(client.get_collections())

collections=[CollectionDescription(name='policy_chunks_test')]


## 데이터 저장 (point 만들기)
> `chunk_id`로 문장·꼬리표(documents)와 벡터(vectors)를 짝지어 **point(번호 + 벡터 + payload)**로 만들고 100개씩 컬렉션에 저장한다.
>> 번호는 chunk_id로 만든 고정 UUID라 다시 실행해도 중복 저장되지 않는다.

In [8]:
import uuid
from qdrant_client.models import PointStruct

# 1) 청크마다 point(번호 + 벡터 + 문장/꼬리표) 만들기
points = []
for doc in documents:
    cid = doc.metadata["chunk_id"]
    points.append(PointStruct(
        id=str(uuid.uuid5(uuid.NAMESPACE_URL, cid)),
        vector=vectors[cid],
        payload={"page_content": doc.page_content, "metadata": doc.metadata},
    ))

# 2) 100개씩 나눠서 저장
for i in range(0, len(points), 100):
    batch = points[i:i + 100]
    client.upsert(collection_name=COLLECTION_NAME, points=batch)
    print(f"배치 {i // 100 + 1} 완료: {len(batch)}개")

print("저장된 개수:", client.count(COLLECTION_NAME).count)

배치 1 완료: 6개
저장된 개수: 6


## retriever 만들기 
> 지역 필터(내 지역 + 전국)와 검수 필터(approved)를 걸어서 신청할 수 없는 정책이나 검수 안 된 근거가 나오지 않게 한다.

### vectorstore 연결

In [9]:
from langchain_ollama import OllamaEmbeddings
from langchain_qdrant import QdrantVectorStore

embedding_model = OllamaEmbeddings(model=EMBED_MODEL)

vectorstore = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding=embedding_model,
)
print("vectorstore 연결 완료")

vectorstore 연결 완료


#### 기본

In [10]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

docs = retriever.invoke("혼자 사는 어르신 돌봄 서비스")

for doc in docs:
    print(f"{doc.metadata['region_name']} | {doc.page_content[:40]}...")

전국 | [전국] 노인맞춤돌봄서비스. 대상: 65세 이상 독거·조손·고령부부 가구...
서울특별시 종로구 | [서울 종로구] 어르신 병원 동행 서비스. 대상: 종로구 65세 이상 혼...
경기도 수원시 | [경기 수원시] 독거노인 생활지원 서비스. 대상: 수원시에 사는 65세 ...


#### 지역 필터

In [11]:
from qdrant_client.models import Filter, FieldCondition, MatchValue, MatchAny

region_filter = Filter(must=[
    FieldCondition(key="metadata.region_code", match=MatchAny(any=["41110", "00000"])),
    FieldCondition(key="metadata.review_status", match=MatchValue(value="approved")),
])

filtered_retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3, "filter": region_filter}
)

docs = filtered_retriever.invoke("혼자 사는 어르신 돌봄 서비스")

for doc in docs:
    print(f"{doc.metadata['region_name']} | {doc.metadata['review_status']} | {doc.page_content[:40]}...")

경기도 수원시 | approved | [경기 수원시] 독거노인 생활지원 서비스. 대상: 수원시에 사는 65세 ...
경기도 수원시 | approved | [경기 수원시] 어르신 무료 급식 지원. 대상: 수원시 60세 이상 저소...
전국 | approved | [전국] 기초연금. 대상: 만 65세 이상이고 소득인정액이 선정기준액 이...


## 검색 함수 (팀 전달용)
> 질문 + 지역번호를 넣으면 근거 정책(문장·점수·정책번호·출처)을 돌려준다. LLM 연결 담당이 이 함수만 불러 쓰면 된다.

In [12]:
def search_policies(question, region_code, k=3):
    my_filter = Filter(must=[
        FieldCondition(key="metadata.region_code", match=MatchAny(any=[region_code, "00000"])),
        FieldCondition(key="metadata.review_status", match=MatchValue(value="approved")),
    ])
    results = vectorstore.similarity_search_with_score(question, k=k, filter=my_filter)

    answer = []
    for doc, score in results:
        answer.append({
            "text": doc.page_content,
            "score": round(score, 3),
            "policy_id": doc.metadata.get("policy_id"),
            "source_name": doc.metadata.get("source_name"),
            "page": doc.metadata.get("page"),
        })
    return answer

search_policies("수원 72세 독거노인 생활비 지원", "41110")

[{'text': '[경기 수원시] 독거노인 생활지원 서비스. 대상: 수원시에 사는 65세 이상 혼자 사는 어르신. 내용: 생활관리사가 안부 확인과 생활 지원을 합니다. 신청: 주소지 행정복지센터.',
  'score': 0.694,
  'policy_id': 'POL-41110-0001',
  'source_name': '지자체복지서비스 API',
  'page': None},
 {'text': '[경기 수원시] 어르신 무료 급식 지원. 대상: 수원시 60세 이상 저소득 어르신. 내용: 경로식당에서 점심 무료 제공. 신청: 가까운 복지관.',
  'score': 0.575,
  'policy_id': 'POL-41110-0002',
  'source_name': '지자체복지서비스 API',
  'page': None},
 {'text': '[전국] 기초연금. 대상: 만 65세 이상이고 소득인정액이 선정기준액 이하인 어르신. 내용: 매달 연금을 지급하여 생활 안정을 돕습니다. 신청: 주소지 행정복지센터 또는 국민연금공단 지사.',
  'score': 0.45,
  'policy_id': None,
  'source_name': '보건복지부 노인복지사업안내(예시)',
  'page': 12}]

## 저장된 컬렉션 다시 불러오기
> Qdrant에 저장된 데이터는 노트북을 꺼도 남아 있으므로, 다음부터는 적재 없이 이 셀만으로 검색할 수 있다.

In [13]:
vectorstore = QdrantVectorStore.from_existing_collection(
    embedding=embedding_model,
    url="http://localhost:6333",
    collection_name=COLLECTION_NAME,
)

print(vectorstore.similarity_search("기초연금", k=1)[0].page_content[:50])

[전국] 기초연금. 대상: 만 65세 이상이고 소득인정액이 선정기준액 이하인 어르신. 내용
